### RAG Data Ingestion Pipeline

In [23]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [24]:
### Read all the pdf files from the directory and convert into Document Structure using Document Loaders.

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all pdfs recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF Files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Every page in a PDF is created as a separate document
            # Add source information to Meta Data
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"Total documents loaded so far: {len(all_documents)}")
    return all_documents    

all_pdf_documents = process_all_pdfs("../data/pdf")

Multiple definitions in dictionary at byte 0x3a550 for key /Creator
Multiple definitions in dictionary at byte 0x3a569 for key /Producer


Found 2 PDF Files to process
Processing Network Programming.pdf
Loaded 47 pages
Processing MYSQL practice 1.pdf
Loaded 12 pages
Total documents loaded so far: 59


In [25]:
### Chunking using Text Splitters

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [26]:
chunks=split_documents(all_pdf_documents)

Split 59 documents into 164 chunks

Example chunk:
Content: Beej’s Guide to Network Programming
Using Internet Sockets
Brian "Beej" Hall
beej@piratehaven.org
Copyright © 1995-2001 by Brian "Beej" Hall
Revision History
Revision Version 1.0.0 August, 1995 Revise...
Metadata: {'producer': 'pdfTeX13.d', 'creator': 'LaTeX with hyperref package', 'creationdate': 'D:20010503022300', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'source': '../data/pdf/Network Programming.pdf', 'total_pages': 47, 'page': 0, 'page_label': '1', 'source_file': 'Network Programming.pdf', 'file_type': 'pdf'}


In [27]:
### Text to Embedding using Sentence Transformers

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
class EmbeddingManager:

    """Handles embedding generation using Sentence Transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model"""
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding Dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        if not self.model:
            raise ValueError("Model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7261.22it/s]


Model loaded successfully. Embedding Dimension: 384


In [29]:
texts=[doc.page_content for doc in chunks]

embeddings=embedding_manager.generate_embeddings(texts)

Generating embeddings for 164 texts
Generated embeddings with shape: (164, 384)


In [31]:
### Storing Embeddings in Vector Database (ChromaDB)

#VectoreStore
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 164


In [32]:
vectorstore.add_documents(chunks,embeddings)

Adding 164 documents to vector store...
Successfully added 164 documents to vector store
Total documents in collection: 328


### RAG Data Retrieval Pipeline

In [33]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [34]:
rag_retriever.retrieve("What is Socket Programming?")

Retrieving documents for query: 'What is Socket Programming?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_4f54efe8_17',
  'content': 'This guide may be freely translated into any language, provided the translation is accurate, and the guide is reprinted\nin its entirety. The translation may also include the name and contact information for the translator.\nThe C source code presented in this document is hereby granted to the public domain.\nContact <beej@piratehaven.org> for more information.\n2. What is a socket?\nYou hear talk of "sockets" all the time, and perhaps you are wondering just what they are exactly. Well, they’re this: a\nway to speak to other programs using standard Unix ﬁle descriptors.\nWhat?\nOk–you may have heard some Unix hacker state, "Jeez, everything in Unix is a ﬁle!" What that person may have\nbeen talking about is the fact that when Unix programs do any sort of I/O, they do it by reading or writing to a ﬁle\ndescriptor. A ﬁle descriptor is simply an integer associated with an open ﬁle. But (and here’s the catch), that ﬁle can',
  'metadata': {'creation

### Generate Response using LLM (Groq API)

In [35]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv('../.env')

### Initialize ChatGroq (Groq API key is stored in .env file)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(api_key=groq_api_key, model="llama-3.3-70b-versatile", temperature=0.7, max_tokens=500)

### Simple RAG
def simple_rag(query: str, retriever: RAGRetriever, llm: ChatGroq):
    """Perform a simple RAG operation: retrieve documents and generate a response"""
    retrieved_docs = retriever.retrieve(query, top_k=5, score_threshold=0.0)
    
    if not retrieved_docs:
        return "No relevant documents found."
    
    # Concatenate retrieved document contents
    context = "\n\n".join([doc['content'] for doc in retrieved_docs])
    
    # Create prompt for LLM
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    # Generate response using LLM
    response = llm.invoke(prompt)
    
    return response

In [36]:
answer = simple_rag("What is Socket Programming?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is Socket Programming?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
content='Socket programming is a way to communicate between two programs or devices using standard Unix file descriptors. It allows programs to read and write to a file descriptor, which can be associated with a file, but also with a network connection, enabling communication with other programs or devices over a network. In other words, socket programming is a way for programs to "speak" to each other using standard Unix file descriptors.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 1101, 'total_tokens': 1183, 'completion_time': 0.310372917, 'completion_tokens_details': None, 'prompt_time': 0.140583535, 'prompt_tokens_details': None, 'queue_time': 0.058852962, 'total_time': 0.450956452}, 'model_name': 'llama-3.3-70b-versatile'